In [1]:
from pathlib import Path
import pandas as pd
import json

BASE_DIR = Path(
    r"D:\DATA SCIENCE COURSE\ML\ML_Projects\WasteWise-AI"
)

RAW_DIR = BASE_DIR / "datasets" / "raw"
OUT_DIR = BASE_DIR / "datasets" / "processed"
REPORT_DIR = BASE_DIR / "datasets" / "reports"

OUT_DIR.mkdir(parents=True, exist_ok=True)
REPORT_DIR.mkdir(parents=True, exist_ok=True)

print("Raw folder exists:", RAW_DIR.exists())

Raw folder exists: True


In [2]:
file_path = RAW_DIR / "calendar.csv"

df = pd.read_csv(file_path)

print("Dataset loaded successfully!")
print("Shape:", df.shape)

Dataset loaded successfully!
Shape: (1969, 14)


In [3]:
display(df.head())

print("Columns:")
print(df.columns.tolist())

print("\nData types:")
print(df.dtypes)

print("\nMissing values:")
print(df.isnull().sum())

print("\nExact duplicate rows:", df.duplicated().sum())

,date,wm_yr_wk,weekday,wday,month,year,d,event_name_1,event_type_1,event_name_2,event_type_2,snap_CA,snap_TX,snap_WI
0,2011-01-29,11101,Saturday,1,1,2011,d_1,NaN,NaN,NaN,NaN,0,0,0
1,2011-01-30,11101,Sunday,2,1,2011,d_2,NaN,NaN,NaN,NaN,0,0,0
2,2011-01-31,11101,Monday,3,1,2011,d_3,NaN,NaN,NaN,NaN,0,0,0
3,2011-02-01,11101,Tuesday,4,2,2011,d_4,NaN,NaN,NaN,NaN,1,1,0
4,2011-02-02,11101,Wednesday,5,2,2011,d_5,NaN,NaN,NaN,NaN,1,0,1


Columns:
['date', 'wm_yr_wk', 'weekday', 'wday', 'month', 'year', 'd', 'event_name_1', 'event_type_1', 'event_name_2', 'event_type_2', 'snap_CA', 'snap_TX', 'snap_WI']

Data types:
date              str
wm_yr_wk        int64
weekday           str
wday            int64
month           int64
year            int64
d                 str
event_name_1      str
event_type_1      str
event_name_2      str
event_type_2      str
snap_CA         int64
snap_TX         int64
snap_WI         int64
dtype: object

Missing values:
date               0
wm_yr_wk           0
weekday            0
wday               0
month              0
year               0
d                  0
event_name_1    1807
event_type_1    1807
event_name_2    1964
event_type_2    1964
snap_CA            0
snap_TX            0
snap_WI            0
dtype: int64

Exact duplicate rows: 0


In [4]:
date_candidates = [
    column for column in df.columns
    if "date" in column.lower()
]

print("Possible date columns:", date_candidates)

Possible date columns: ['date']


In [5]:
df["date"] = pd.to_datetime(
    df["date"],
    errors="coerce"
)

print("Date data type:", df["date"].dtype)
print("Invalid dates:", df["date"].isna().sum())

display(df[["date"]].head())

Date data type: datetime64[us]
Invalid dates: 0


,date
0,2011-01-29
1,2011-01-30
2,2011-01-31
3,2011-02-01
4,2011-02-02


In [6]:
print("Earliest date:", df["date"].min())
print("Latest date:", df["date"].max())

print("Number of unique dates:", df["date"].nunique())

Earliest date: 2011-01-29 00:00:00
Latest date: 2016-06-19 00:00:00
Number of unique dates: 1969


In [7]:
duplicate_date_mask = df["date"].duplicated(
    keep=False
)

duplicate_dates = df[duplicate_date_mask].copy()

print("Rows with duplicate dates:", len(duplicate_dates))


display(duplicate_dates.head())

Rows with duplicate dates: 0


,date,wm_yr_wk,weekday,wday,month,year,d,event_name_1,event_type_1,event_name_2,event_type_2,snap_CA,snap_TX,snap_WI


In [8]:
invalid_date_mask = df["date"].isna()

rejected_df = df[invalid_date_mask].copy()
clean_df = df[~invalid_date_mask].copy()

print("Original rows:", len(df))
print("Rows with invalid dates:", len(rejected_df))
print("Rows remaining:", len(clean_df))

Original rows: 1969
Rows with invalid dates: 0
Rows remaining: 1969


In [9]:
before = len(clean_df)

clean_df = clean_df.drop_duplicates().copy()

print("Exact duplicate rows removed:", before - len(clean_df))
print("Rows after duplicate removal:", len(clean_df))

Exact duplicate rows removed: 0
Rows after duplicate removal: 1969


In [10]:
duplicate_date_mask = clean_df["date"].duplicated(
    keep=False
)

duplicate_dates = clean_df[duplicate_date_mask].copy()

clean_df = clean_df[~duplicate_date_mask].copy()

clean_df = clean_df.sort_values(
    "date"
).reset_index(drop=True)

print("Duplicate-date rows for review:", len(duplicate_dates))
print("Final clean rows:", len(clean_df))

Duplicate-date rows for review: 0
Final clean rows: 1969


In [11]:
clean_path = OUT_DIR / "calendar_clean.csv"
rejected_path = REPORT_DIR / "calendar_rejected_dates.csv"
duplicate_path = REPORT_DIR / "calendar_duplicate_dates.csv"
report_path = REPORT_DIR / "calendar_report.json"

clean_df.to_csv(clean_path, index=False)
rejected_df.to_csv(rejected_path, index=False)
duplicate_dates.to_csv(duplicate_path, index=False)

report = {
    "original_rows": int(len(df)),
    "clean_rows": int(len(clean_df)),
    "invalid_date_rows": int(len(rejected_df)),
    "duplicate_date_rows": int(len(duplicate_dates)),
    "columns": clean_df.columns.tolist(),
    "earliest_date": str(clean_df["date"].min()),
    "latest_date": str(clean_df["date"].max())
}

with open(report_path, "w", encoding="utf-8") as file:
    json.dump(report, file, indent=4)

print("Clean calendar:", clean_path)
print("Rejected dates:", rejected_path)
print("Duplicate dates:", duplicate_path)
print("Cleaning report:", report_path)

Clean calendar: D:\DATA SCIENCE COURSE\ML\ML_Projects\WasteWise-AI\datasets\processed\calendar_clean.csv
Rejected dates: D:\DATA SCIENCE COURSE\ML\ML_Projects\WasteWise-AI\datasets\reports\calendar_rejected_dates.csv
Duplicate dates: D:\DATA SCIENCE COURSE\ML\ML_Projects\WasteWise-AI\datasets\reports\calendar_duplicate_dates.csv
Cleaning report: D:\DATA SCIENCE COURSE\ML\ML_Projects\WasteWise-AI\datasets\reports\calendar_report.json


In [12]:
print("Final shape:", clean_df.shape)
print("Missing dates:", clean_df["date"].isna().sum())
print("Duplicate dates:", clean_df["date"].duplicated().sum())

display(clean_df.head())

Final shape: (1969, 14)
Missing dates: 0
Duplicate dates: 0


,date,wm_yr_wk,weekday,wday,month,year,d,event_name_1,event_type_1,event_name_2,event_type_2,snap_CA,snap_TX,snap_WI
0,2011-01-29,11101,Saturday,1,1,2011,d_1,NaN,NaN,NaN,NaN,0,0,0
1,2011-01-30,11101,Sunday,2,1,2011,d_2,NaN,NaN,NaN,NaN,0,0,0
2,2011-01-31,11101,Monday,3,1,2011,d_3,NaN,NaN,NaN,NaN,0,0,0
3,2011-02-01,11101,Tuesday,4,2,2011,d_4,NaN,NaN,NaN,NaN,1,1,0
4,2011-02-02,11101,Wednesday,5,2,2011,d_5,NaN,NaN,NaN,NaN,1,0,1
